In [52]:
import torch

gates = torch.tensor([
    [0.0, 0.7, 0.0],
    [0.1, 0.0, 0.4],
    [0.0, 0.0, 0.9]
])
gate = torch.eye(32)

In [53]:
sorted_experts, index_sorted_experts = torch.nonzero(gates).sort(0)

In [54]:
sorted_experts

tensor([[0, 0],
        [1, 1],
        [1, 2],
        [2, 2]])

In [55]:
index_sorted_experts

tensor([[0, 1],
        [1, 0],
        [2, 2],
        [3, 3]])

In [56]:
sorted_experts.split(1,dim=1)

(tensor([[0],
         [1],
         [1],
         [2]]),
 tensor([[0],
         [1],
         [2],
         [2]]))

In [57]:
_,_expert_index = sorted_experts.split(1,dim=1)
_expert_index.shape

torch.Size([4, 1])

In [58]:
index_sorted_experts

tensor([[0, 1],
        [1, 0],
        [2, 2],
        [3, 3]])

In [59]:
index_sorted_experts[:,1]

tensor([1, 0, 2, 3])

In [60]:
torch.nonzero(gates)

tensor([[0, 1],
        [1, 0],
        [1, 2],
        [2, 2]])

In [61]:
_batch_index = torch.nonzero(gates)[index_sorted_experts[:,1],0]
_batch_index

tensor([1, 0, 1, 2])

In [62]:
(gate > 0).sum().tolist()

32

In [63]:
_batch_index.flatten()

tensor([1, 0, 1, 2])

In [64]:
gates

tensor([[0.0000, 0.7000, 0.0000],
        [0.1000, 0.0000, 0.4000],
        [0.0000, 0.0000, 0.9000]])

In [65]:
gates_exp=gates[_batch_index.flatten()]
gates_exp

tensor([[0.1000, 0.0000, 0.4000],
        [0.0000, 0.7000, 0.0000],
        [0.1000, 0.0000, 0.4000],
        [0.0000, 0.0000, 0.9000]])

In [66]:
gates_exp.shape

torch.Size([4, 3])

In [67]:
_expert_index

tensor([[0],
        [1],
        [2],
        [2]])

In [68]:
torch.arange(10)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [69]:
_expert_index

tensor([[0],
        [1],
        [2],
        [2]])

In [70]:
gates_exp

tensor([[0.1000, 0.0000, 0.4000],
        [0.0000, 0.7000, 0.0000],
        [0.1000, 0.0000, 0.4000],
        [0.0000, 0.0000, 0.9000]])

In [71]:
torch.gather(gates_exp, 1, _expert_index).shape

torch.Size([4, 1])

In [72]:
gates_exp.shape

torch.Size([4, 3])

In [73]:
torch.unsqueeze(gates_exp,1).shape

torch.Size([4, 1, 3])

In [74]:
X = torch.rand(4,10)
X

tensor([[0.9896, 0.9776, 0.1296, 0.4235, 0.2581, 0.0094, 0.2701, 0.0399, 0.0392,
         0.1016],
        [0.9072, 0.3602, 0.2366, 0.3511, 0.5168, 0.1960, 0.3510, 0.8475, 0.8123,
         0.7575],
        [0.6749, 0.2354, 0.2323, 0.3100, 0.1054, 0.9786, 0.1576, 0.5135, 0.1149,
         0.4260],
        [0.5750, 0.7163, 0.9979, 0.5819, 0.2086, 0.9800, 0.7967, 0.4230, 0.1518,
         0.3035]])

In [75]:
_batch_index

tensor([1, 0, 1, 2])

In [76]:
X=X[_batch_index].squeeze(1)
X.shape

torch.Size([4, 10])

In [77]:
Q=torch.split(X,3,dim=0)
Q

(tensor([[0.9072, 0.3602, 0.2366, 0.3511, 0.5168, 0.1960, 0.3510, 0.8475, 0.8123,
          0.7575],
         [0.9896, 0.9776, 0.1296, 0.4235, 0.2581, 0.0094, 0.2701, 0.0399, 0.0392,
          0.1016],
         [0.9072, 0.3602, 0.2366, 0.3511, 0.5168, 0.1960, 0.3510, 0.8475, 0.8123,
          0.7575]]),
 tensor([[0.6749, 0.2354, 0.2323, 0.3100, 0.1054, 0.9786, 0.1576, 0.5135, 0.1149,
          0.4260]]))

In [78]:
from moe import MoE
import torch

# instantiate the MoE layer
model = MoE(input_size=1000, output_size=20, num_experts=10,hidden_size=66, k= 2, noisy_gating=True)

X = torch.rand(32, 1000)
print(X.shape)

#train
model.train()
# forward
y_hat, aux_loss = model(X)

# evaluation

model.eval()
y_hat, aux_loss = model(X)
print(f"y_hat: {y_hat[1]} and loss: {aux_loss}")


torch.Size([32, 1000])
y_hat: tensor([0.0447, 0.0491, 0.0471, 0.0499, 0.0574, 0.0447, 0.0570, 0.0464, 0.0493,
        0.0496, 0.0587, 0.0554, 0.0449, 0.0455, 0.0503, 0.0443, 0.0487, 0.0562,
        0.0479, 0.0530], grad_fn=<SelectBackward0>) and loss: 0.08888889104127884


In [79]:
torch.randn(2,2)

tensor([[ 0.5498, -1.8988],
        [ 0.2776, -0.3215]])

In [80]:
import torch.nn as nn
x = torch.tensor([[1,2],[3,4]], dtype=torch.float32)
print(x)
soft = nn.Softplus()
ans= torch.rand_like(x)
print(ans)

tensor([[1., 2.],
        [3., 4.]])
tensor([[0.9771, 0.8076],
        [0.9187, 0.3564]])


In [81]:
d = torch.tensor([[1,2],[3,4]])
b = torch.tensor([[3,4],[5,6]])
print(d)
print(b)
c=d.sum(0, keepdim=True)
print(c)

tensor([[1, 2],
        [3, 4]])
tensor([[3, 4],
        [5, 6]])
tensor([[4, 6]])


In [82]:
a = torch.rand(32,10)
a

tensor([[7.1523e-01, 8.2477e-01, 1.5270e-01, 3.8851e-01, 5.7476e-01, 2.9202e-01,
         6.5277e-01, 3.7354e-04, 5.1935e-01, 7.3555e-01],
        [1.1511e-01, 1.0632e-01, 7.8514e-02, 7.2295e-01, 8.0015e-01, 3.2685e-01,
         9.2556e-02, 9.1399e-01, 6.9014e-01, 6.0305e-01],
        [3.1334e-01, 2.1401e-01, 5.1854e-01, 9.9952e-01, 5.3283e-01, 1.9380e-01,
         2.7859e-01, 4.3808e-01, 3.8208e-01, 3.6150e-01],
        [5.5098e-01, 8.5265e-01, 5.2179e-01, 4.0527e-01, 7.0512e-01, 6.0410e-01,
         5.5911e-01, 3.2514e-01, 9.5829e-01, 2.8783e-01],
        [8.1149e-01, 5.5935e-01, 5.3176e-01, 8.1331e-01, 6.5400e-01, 2.7578e-01,
         4.3529e-01, 4.6483e-01, 8.2149e-01, 2.0772e-01],
        [1.2223e-01, 9.7453e-01, 3.1967e-01, 8.9375e-01, 3.8766e-02, 4.9138e-01,
         5.4683e-01, 8.5354e-01, 2.5045e-01, 6.8179e-01],
        [6.3955e-01, 6.1662e-01, 4.4946e-01, 9.0094e-01, 9.7470e-01, 6.6253e-01,
         9.3446e-03, 3.2843e-01, 5.5510e-01, 9.5632e-01],
        [9.1368e-01, 9.8667

In [83]:
q,w=a.topk(1,dim=1)
q

tensor([[0.8248],
        [0.9140],
        [0.9995],
        [0.9583],
        [0.8215],
        [0.9745],
        [0.9747],
        [0.9867],
        [0.9446],
        [0.9401],
        [0.8395],
        [0.9348],
        [0.9615],
        [0.9928],
        [0.9728],
        [0.9549],
        [0.8811],
        [0.9975],
        [0.9127],
        [0.9471],
        [0.8163],
        [0.8966],
        [0.9862],
        [0.8119],
        [0.8757],
        [0.9761],
        [0.9483],
        [0.8161],
        [0.8036],
        [0.9978],
        [0.9643],
        [0.9748]])

In [84]:
# z=q[:,:4]
z = torch.tensor([[1,2,1],[3,4,1],[6,7,1]])
w = z.sum(0)
print(w)
w = w.float().mean()
print(w)
v = z.sum(1, keepdim=True)
print(v.shape)
out=z/(z.sum(1, keepdim=True))
out

tensor([10, 13,  3])
tensor(8.6667)
torch.Size([3, 1])


tensor([[0.2500, 0.5000, 0.2500],
        [0.3750, 0.5000, 0.1250],
        [0.4286, 0.5000, 0.0714]])

In [85]:
a= torch.arange(32)

In [86]:
f=torch.rand(32,4)
f=f.flatten()
v = torch.unsqueeze(torch.gather(f,0,a),1)
v

tensor([[0.2516],
        [0.6406],
        [0.6560],
        [0.6059],
        [0.0376],
        [0.7292],
        [0.6115],
        [0.4080],
        [0.5682],
        [0.0358],
        [0.8815],
        [0.2572],
        [0.3517],
        [0.7191],
        [0.3996],
        [0.7185],
        [0.9866],
        [0.7967],
        [0.1576],
        [0.1741],
        [0.2635],
        [0.9777],
        [0.4404],
        [0.4195],
        [0.9520],
        [0.7937],
        [0.6630],
        [0.7557],
        [0.5729],
        [0.9834],
        [0.3329],
        [0.5504]])

In [87]:
n = torch.rand(32,10)
print(n[0])
m=n-v
m[0]

tensor([0.7024, 0.9588, 0.2397, 0.6618, 0.6484, 0.2218, 0.1299, 0.6672, 0.7933,
        0.9121])


tensor([ 0.4508,  0.7072, -0.0119,  0.4102,  0.3968, -0.0298, -0.1217,  0.4156,
         0.5417,  0.6605])

In [88]:
g=(torch.arange(32)*4)+3
g

tensor([  3,   7,  11,  15,  19,  23,  27,  31,  35,  39,  43,  47,  51,  55,
         59,  63,  67,  71,  75,  79,  83,  87,  91,  95,  99, 103, 107, 111,
        115, 119, 123, 127])

In [89]:
g-1

tensor([  2,   6,  10,  14,  18,  22,  26,  30,  34,  38,  42,  46,  50,  54,
         58,  62,  66,  70,  74,  78,  82,  86,  90,  94,  98, 102, 106, 110,
        114, 118, 122, 126])

In [90]:
torch.unsqueeze(torch.gather(f,0,g),1).shape

torch.Size([32, 1])

In [91]:
from torch.distributions import Normal
normal = Normal(0,1)
a=torch.rand(2,2)
b= torch.rand(2,2)
c=normal.cdf(a-b)
print(c)
# c= torch.gt(b,a)
d=torch.rand(2,2)
e= torch.rand(2,2)
f=normal.cdf(d-e)
print(f)

tensor([[0.3929, 0.5302],
        [0.2110, 0.5407]])
tensor([[0.2933, 0.5020],
        [0.5668, 0.5214]])


In [92]:
a=torch.tensor([[True,False],[False,False]])
z=torch.where(a,c,f)
z==f

tensor([[False,  True],
        [ True,  True]])

In [93]:
a= torch.tensor([[1,2],[2,3],[7,5]])
print(a)
torch.max(a,dim=-1)

tensor([[1, 2],
        [2, 3],
        [7, 5]])


torch.return_types.max(
values=tensor([2, 3, 7]),
indices=tensor([1, 1, 0]))

In [94]:
import torch

In [95]:
a = torch.randn(5,2)
a

tensor([[ 0.0746, -1.2965],
        [ 1.7478,  1.6296],
        [ 0.1322, -0.2162],
        [ 1.5903,  0.2075],
        [-1.2098,  0.7428]])

In [96]:
c,d=torch.max(a,dim=-1)
c,d

(tensor([0.0746, 1.7478, 0.1322, 1.5903, 0.7428]), tensor([0, 0, 0, 0, 1]))

In [97]:
indexes_list = [torch.eq(d, i).nonzero(as_tuple=True)[0] for i in range(2)]
indexes_list

[tensor([0, 1, 2, 3]), tensor([4])]

In [98]:
torch.eq(d,1)

tensor([False, False, False, False,  True])

In [99]:
import torch
a =torch.rand(2,3,10)
b = torch.rand(10,3)
c = a @ b
c

tensor([[[2.0320, 2.9005, 2.3913],
         [2.2764, 2.7007, 2.1394],
         [1.7904, 2.1094, 2.0775]],

        [[2.0525, 2.6344, 2.1862],
         [1.4936, 2.3073, 2.0069],
         [2.2654, 2.7581, 2.6318]]])

In [100]:
a = torch.rand(2,3,2)
b = torch.rand(2,3,2)
c = torch.rand(2,3,2)

In [101]:
B , T, C = a.size()
head = 2

In [102]:
a = a.view(B,T,head,C//head).transpose(1,2) # 2,3,2,2//2
b = b.view(B,T,head,C//head).transpose(1,2) # 2,3,2,2//2
c = c.view(B,T,head,C//head).transpose(1,2) # 2,3,2,2//2

In [103]:
import math
import torch.nn.functional as F

bias = torch.tril(torch.ones(T, T)).view(1, 1, T, T).to(a.device)
att = att = (a @ b.transpose(-2,-1)) * (1.0 / math.sqrt(b.size(-1)))
att = att.masked_fill(bias == 0, float('-inf'))
att = F.softmax(att, dim=-1)
y = att @ c

In [104]:
y=y.transpose(1,2).contiguous().view(B, T, C)
y.shape

torch.Size([2, 3, 2])

In [105]:
net = torch.nn.Sequential(
    torch.nn.Linear(2,2)
)

In [106]:
# net.zero_grad()
yhat = net(y)

In [107]:
yhat.shape

torch.Size([2, 3, 2])

In [108]:
class DecoderHead(nn.Module):
    """one head of self-attention"""
    def __init__(self, n_embd, head_size, block_size=1024):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.2)

    def forward(self, embedding):
        B,T,C = embedding.shape
        k = self.key(embedding)
        # print(k.shape)
        # print(k)
        q = self.query(embedding)
        # print(q.shape)
        # print(q)
        scale = q.size(-1)**0.5
        wei = (q @ k.transpose(-2,-1))/scale
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(embedding)
        out = wei @ v

        return out


In [ ]:
x = torch.randn(2,3,2)
x

tensor([[[-0.7181, -0.5145],
         [-0.7808,  0.2866],
         [-1.0968, -0.3228]],

        [[-0.8859,  0.0670],
         [-0.1423, -0.3474],
         [ 0.2703, -0.0761]]])

In [ ]:
x

In [111]:
m1 = DecoderHead(2,1)
m1(x)

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
m1.zero_grad()
yhat = m1(x)


In [1]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")


In [2]:
enc.n_vocab

100277